# 🚀 Praca Magisterska - Google Colab Setup
Ten notatnik pozwoli Ci sklonować repozytorium i uruchamiać eksperymenty RecBole na darmowym GPU Google Colab.

### ⚠️ Instrukcja:
1. Upewnij się, że w Colab masz włączone GPU: `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> `T4 GPU`.

In [ ]:
# 1. Klonowanie repozytorium z GitHub
!git clone https://github.com/Sornat11/Praca_magisterska.git

# 2. Przejście do folderu projektu
%cd Praca_magisterska

print("\nZawartość katalogu:")
!ls

In [ ]:
# 3. Instalacja niezbędnych bibliotek
!pip install recbole hyperopt ray[tune] pandas pyyaml

# 4. Aplikowanie patcha kompatybilności dla SciPy w bibliotece RecBole
import recbole, os
path = os.path.join(os.path.dirname(recbole.__file__), 'model', 'general_recommender', 'lightgcn.py')
content = open(path).read().replace('A._update(data_dict)', 'for k, v in data_dict.items(): A[k] = v')
open(path, 'w').write(content)
print("Patch dla SciPy 1.11+ został zaaplikowany pomyślnie!")

In [ ]:
# 5. Pobranie danych MovieLens i konwersja do formatu RecBole
!python 1_Preprocessing/import_raw_data.py
!python 1_Preprocessing/convert_to_recbole.py

## 🧪 Uruchamianie Eksperymentów
Możesz uruchamiać skrypty treningowe dla poszczególnych modeli.

In [ ]:
!python 2_Experiments/run_experiment.py --model BPR --dataset ml-100k --config 2_Experiments/Configs/bpr.yaml

In [ ]:
!python 2_Experiments/run_experiment.py --model NCF --dataset ml-100k --config 2_Experiments/Configs/ncf.yaml

In [ ]:
!python 2_Experiments/run_experiment.py --model LightGCN --dataset ml-100k --config 2_Experiments/Configs/gnn.yaml

## ⚡ Optymalizacja Hiperparametrów (HPO)
Skrypty do uruchomienia automatycznego dostrajania parametrów (RecBole HyperTuning).
*Uwaga: Optymalizacja LightGCN jest ekstremalnie czasochłonna na CPU.*

In [ ]:
# Optymalizacja BPR-MF
!python 2_Experiments/run_hyper.py --model BPR --config 2_Experiments/Configs/bpr.yaml --hyper 2_Experiments/Hyperparams/bpr.hyper

# Optymalizacja NCF
# !python 2_Experiments/run_hyper.py --model NCF --config 2_Experiments/Configs/ncf.yaml --hyper 2_Experiments/Hyperparams/ncf.hyper

# Optymalizacja LightGCN
# !python 2_Experiments/run_hyper.py --model LightGCN --config 2_Experiments/Configs/gnn.yaml --hyper 2_Experiments/Hyperparams/gnn.hyper

## 📊 Agregacja wyników
Po zakończeniu obliczeń, uruchom ten kod, aby zaktualizować plik Excel z wynikami.

In [ ]:
!python 3_Evaluation/aggregate_results.py